# Colab Local LLM Classifier
Runs the classifier stage locally on Colab with Hugging Face Transformers and writes classifier-only prediction parquet files to Drive.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/data/llm-gatekeeping'
REPO_URL = 'https://github.com/noamdwc/llm-gatekeeping.git'
REPO_DIR = '/content/llm-gatekeeping'
BRANCH = 'zero_shot_nlp_attack'

MAIN_SPLITS = ['train', 'val', 'test', 'unseen_val', 'unseen_test', 'safeguard_test']
EXTERNAL_DATASETS = ['deepset', 'jackhhao']
TARGET_LIMIT = 5  # Set to None for every row in every target.
MODEL_ID = 'meta/llama-3.1-8b-instruct'
HF_MODEL_ID = 'meta-llama/Llama-3.1-8B-Instruct'
MODEL_PROVIDER_NAME = 'transformers-local'
MAX_MODEL_LEN = 4096
BATCH_SIZE = 1
CHECKPOINT_EVERY = 50
OUTPUT_SUFFIX = 'colab_local_classifier'

HF_CACHE_DIR = f'{DRIVE_ROOT}/cache/huggingface'
SPLITS_DIR = f'{DRIVE_ROOT}/data/processed/splits'
PREDICTIONS_DIR = f'{DRIVE_ROOT}/data/processed/predictions'
PREDICTIONS_EXTERNAL_DIR = f'{DRIVE_ROOT}/data/processed/predictions_external'

import os

os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['HF_HUB_CACHE'] = f'{HF_CACHE_DIR}/hub'
os.environ['TRANSFORMERS_CACHE'] = f'{HF_CACHE_DIR}/transformers'
os.environ['HF_DATASETS_CACHE'] = f'{HF_CACHE_DIR}/datasets'
print(f'Hugging Face cache: {HF_CACHE_DIR}')
print(f'Main predictions dir: {PREDICTIONS_DIR}')
print(f'External predictions dir: {PREDICTIONS_EXTERNAL_DIR}')


In [ ]:
import os
import subprocess

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print(f'Using existing repo at {REPO_DIR}')

os.chdir(REPO_DIR)
print('Repo:', os.getcwd())

subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], check=True)

In [ ]:
%pip install -q -r requirements.txt
%pip install -q "transformers>=4.36" "accelerate>=0.25"


In [ ]:
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError('Colab GPU is unavailable. Enable Runtime > Change runtime type > GPU.')

Path(PREDICTIONS_DIR).mkdir(parents=True, exist_ok=True)
Path(PREDICTIONS_EXTERNAL_DIR).mkdir(parents=True, exist_ok=True)
Path(HF_CACHE_DIR).mkdir(parents=True, exist_ok=True)

model_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'Loading tokenizer: {HF_MODEL_ID}')
tokenizer = AutoTokenizer.from_pretrained(
    HF_MODEL_ID,
    cache_dir=HF_CACHE_DIR,
    trust_remote_code=True,
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Loading model with dtype={model_dtype}: {HF_MODEL_ID}')
model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_ID,
    cache_dir=HF_CACHE_DIR,
    torch_dtype=model_dtype,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()
model_device = next(model.parameters()).device
print(f'Transformers model is ready: {MODEL_ID} on {model_device}')


In [ ]:
from pathlib import Path

import pandas as pd

from src.eval_external import load_external_dataset
from src.llm_classifier.constants import NLP_TYPES
from src.llm_classifier.llm_classifier import HierarchicalLLMClassifier
from src.llm_classifier.llm_classifier import build_few_shot_examples
from src.llm_classifier.prompts import build_classifier_messages
from src.utils import build_sample_id, load_config

cfg = load_config()
text_col = cfg['dataset']['text_col']
label_col = cfg['dataset']['label_col']

train_path = Path(SPLITS_DIR) / 'train.parquet'
if not train_path.exists():
    raise FileNotFoundError(f'Missing train split: {train_path}')

train_df = pd.read_parquet(train_path)
missing_train = {text_col, label_col} - set(train_df.columns)
if missing_train:
    raise ValueError(f'{train_path} missing required columns: {sorted(missing_train)}')

few_shot, used_ids = build_few_shot_examples(train_df, cfg)
print(f'Train rows for few-shot source: {len(train_df):,}; few-shot pairs: {len(few_shot)}')


In [ ]:
def make_main_target(split: str) -> dict:
    return {
        'kind': 'main',
        'key': split,
        'split': split,
        'text_col': text_col,
        'label_col': label_col,
        'input_path': Path(SPLITS_DIR) / f'{split}.parquet',
        'checkpoint_path': Path(PREDICTIONS_DIR) / f'llm_checkpoint_{split}_{OUTPUT_SUFFIX}.parquet',
        'output_path': Path(PREDICTIONS_DIR) / f'llm_predictions_{split}_{OUTPUT_SUFFIX}.parquet',
        'progress_label': f'main:{split}',
    }


def make_external_target(dataset_key: str) -> dict:
    return {
        'kind': 'external',
        'key': dataset_key,
        'dataset_key': dataset_key,
        'text_col': 'modified_sample',
        'label_col': 'label_binary',
        'checkpoint_path': Path(PREDICTIONS_EXTERNAL_DIR) / f'llm_checkpoint_external_{dataset_key}_{OUTPUT_SUFFIX}.parquet',
        'output_path': Path(PREDICTIONS_EXTERNAL_DIR) / f'llm_predictions_external_{dataset_key}.parquet',
        'progress_label': f'external:{dataset_key}',
    }


TARGETS = [make_main_target(split) for split in MAIN_SPLITS]
TARGETS.extend(make_external_target(dataset_key) for dataset_key in EXTERNAL_DATASETS)
print('Configured targets:')
for target in TARGETS:
    print(f"- {target['progress_label']} -> {target['output_path']}")


In [ ]:
import json

VALID_NLP_TYPES = set(NLP_TYPES) | {'none'}


def build_static_few_shot_messages(text: str) -> list[dict]:
    messages = []
    for benign_text, attack_text, attack_type in few_shot:
        messages.append({'role': 'user', 'content': f'INPUT_PROMPT:\n{benign_text}'})
        messages.append({
            'role': 'assistant',
            'content': json.dumps({
                'label': 'benign',
                'confidence': 95,
                'nlp_attack_type': 'none',
                'evidence': '',
                'reason': 'No active attempt to override instructions, exfiltrate data, or hijack tools.',
            }),
        })
        if attack_type in NLP_TYPES:
            evidence = ''
            adv_reason = f'Perturbed tokens characteristic of {attack_type} adversarial attack.'
        else:
            evidence = attack_text[:80]
            adv_reason = f'Contains {attack_type} obfuscation; active adversarial prompt detected.'
        messages.append({'role': 'user', 'content': f'INPUT_PROMPT:\n{attack_text}'})
        messages.append({
            'role': 'assistant',
            'content': json.dumps({
                'label': 'adversarial',
                'confidence': 84,
                'nlp_attack_type': attack_type if attack_type in NLP_TYPES else 'none',
                'evidence': evidence,
                'reason': adv_reason,
            }),
        })
    return messages


def format_chat_prompt(messages: list[dict]) -> str:
    if getattr(tokenizer, 'chat_template', None):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    formatted = []
    for message in messages:
        role = str(message.get('role', 'user')).upper()
        content = str(message.get('content', ''))
        formatted.append(f'{role}: {content}')
    formatted.append('ASSISTANT:')
    return '\n\n'.join(formatted)


def extract_generation_logprobs(generated_ids, scores) -> list[dict] | None:
    if generated_ids is None or len(generated_ids) == 0 or not scores:
        return None
    top_k = max(0, int(cfg['llm'].get('top_logprobs', 5)))
    extracted = []
    token_ids = generated_ids.tolist()
    for token_id, score in zip(token_ids, scores):
        log_probs = torch.log_softmax(score[0].detach().float(), dim=-1)
        token_text = tokenizer.decode([token_id], skip_special_tokens=False)
        token_payload = {
            'token': token_text,
            'logprob': float(log_probs[token_id].item()),
            'top_logprobs': [],
        }
        if top_k > 0:
            k = min(top_k, log_probs.numel())
            top_values, top_indices = torch.topk(log_probs, k=k)
            token_payload['top_logprobs'] = [
                {
                    'token': tokenizer.decode([int(alt_id)], skip_special_tokens=False),
                    'logprob': float(alt_logprob.item()),
                }
                for alt_logprob, alt_id in zip(top_values, top_indices)
            ]
        extracted.append(token_payload)
    return extracted or None


def normalize_classifier_payload(payload: dict, raw_response_text: str | None, token_logprobs: list[dict] | None) -> dict:
    if not isinstance(payload, dict) or not payload:
        raw_response_text = raw_response_text or ''
        label = 'adversarial'
        nlp_attack_type = 'none'
        category = HierarchicalLLMClassifier._derive_category(label, nlp_attack_type)
        return {
            'label': label,
            'confidence': 0.0,
            'nlp_attack_type': nlp_attack_type,
            'evidence': 'parse_failure',
            'reason': 'classifier response could not be parsed',
            '_token_logprobs': token_logprobs,
            '_provider_name': MODEL_PROVIDER_NAME,
            '_model_name': MODEL_ID,
            '_raw_response_text': raw_response_text,
            '_parse_success': False,
            '_category': category,
            'clf_label': label,
            'clf_category': category,
            'clf_confidence': 0.0,
            'clf_evidence': 'parse_failure',
            'clf_nlp_attack_type': nlp_attack_type,
            'clf_provider_name': MODEL_PROVIDER_NAME,
            'clf_model_name': MODEL_ID,
            'clf_raw_response_text': raw_response_text,
            'clf_parse_success': False,
            'clf_token_logprobs': token_logprobs,
        }

    label = payload.get('label', '') if payload else ''
    if label not in ('benign', 'adversarial', 'uncertain'):
        label = 'adversarial'

    nlp_attack_type = payload.get('nlp_attack_type', 'none') if payload else 'none'
    if nlp_attack_type not in VALID_NLP_TYPES:
        nlp_attack_type = 'none'
    if label != 'adversarial':
        nlp_attack_type = 'none'

    evidence = payload.get('evidence', '') if payload else ''
    if label != 'adversarial':
        evidence = ''

    confidence = HierarchicalLLMClassifier._normalize_confidence(
        payload.get('confidence', 50) if payload else 50
    )
    category = HierarchicalLLMClassifier._derive_category(label, nlp_attack_type)

    return {
        'label': label,
        'confidence': confidence,
        'nlp_attack_type': nlp_attack_type,
        'evidence': evidence,
        'reason': payload.get('reason', '') if payload else '',
        '_token_logprobs': token_logprobs,
        '_provider_name': MODEL_PROVIDER_NAME,
        '_model_name': MODEL_ID,
        '_raw_response_text': raw_response_text,
        '_parse_success': bool(payload),
        '_category': category,
        'clf_label': label,
        'clf_category': category,
        'clf_confidence': confidence,
        'clf_evidence': evidence,
        'clf_nlp_attack_type': nlp_attack_type,
        'clf_provider_name': MODEL_PROVIDER_NAME,
        'clf_model_name': MODEL_ID,
        'clf_raw_response_text': raw_response_text,
        'clf_parse_success': bool(payload),
        'clf_token_logprobs': token_logprobs,
    }


def parse_classifier_json(raw_response_text: str | None) -> dict:
    if raw_response_text is None:
        return {}
    try:
        payload = json.loads(raw_response_text)
        if isinstance(payload, str):
            payload = json.loads(payload)
        return payload if isinstance(payload, dict) else {}
    except json.JSONDecodeError:
        return {}


def classify_text(text: str) -> dict:
    messages = build_classifier_messages(text, build_static_few_shot_messages(text))
    prompt = format_chat_prompt(messages)
    max_new_tokens = int(cfg['llm']['max_tokens_classifier'])
    max_prompt_tokens = max(1, int(MAX_MODEL_LEN) - max_new_tokens)
    encoded = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=max_prompt_tokens,
    )
    model_inputs = {name: tensor.to(model_device) for name, tensor in encoded.items()}
    temperature = float(cfg['llm'].get('temperature', 0) or 0)
    generation_kwargs = {
        'max_new_tokens': max_new_tokens,
        'return_dict_in_generate': True,
        'output_scores': True,
        'pad_token_id': tokenizer.pad_token_id,
        'eos_token_id': tokenizer.eos_token_id,
    }
    if temperature > 0:
        generation_kwargs['do_sample'] = True
        generation_kwargs['temperature'] = temperature
    else:
        generation_kwargs['do_sample'] = False

    with torch.inference_mode():
        generated = model.generate(**model_inputs, **generation_kwargs)

    input_token_count = model_inputs['input_ids'].shape[-1]
    generated_ids = generated.sequences[0, input_token_count:]
    raw_response_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    token_logprobs = extract_generation_logprobs(generated_ids, generated.scores)
    payload = parse_classifier_json(raw_response_text)
    return normalize_classifier_payload(payload, raw_response_text, token_logprobs)


In [ ]:
GROUND_TRUTH_COLUMNS = [
    'modified_sample',
    'original_sample',
    'attack_name',
    'label_binary',
    'label_category',
    'label_type',
    'prompt_hash',
    'benign_source',
    'is_synthetic_benign',
]
PREDICTION_COLUMNS = [
    'llm_pred_binary',
    'llm_pred_raw',
    'llm_pred_category',
    'llm_conf_binary',
    'llm_evidence',
    'llm_stages_run',
    'llm_provider_name',
    'llm_model_name',
    'llm_raw_response_text',
    'llm_parse_success',
    'clf_label',
    'clf_category',
    'clf_confidence',
    'clf_evidence',
    'clf_nlp_attack_type',
    'clf_provider_name',
    'clf_model_name',
    'clf_raw_response_text',
    'clf_parse_success',
    'clf_token_logprobs',
]


def valid_prediction_mask(df: pd.DataFrame) -> pd.Series:
    required_columns = {'sample_id', *PREDICTION_COLUMNS}
    if not required_columns.issubset(df.columns):
        return pd.Series(False, index=df.index)
    required_non_null_columns = ['sample_id', *PREDICTION_COLUMNS]
    return (
        df[required_non_null_columns].notna().all(axis=1)
        & df['llm_stages_run'].eq(1)
        & df['llm_pred_binary'].isin({'benign', 'adversarial'})
        & df['llm_pred_raw'].isin({'benign', 'adversarial', 'uncertain'})
        & df['llm_conf_binary'].notna()
        & df['clf_confidence'].notna()
        & df['llm_parse_success'].notna()
    )


def make_failure_result(raw_response_text: str | None = None) -> dict:
    return normalize_classifier_payload({}, raw_response_text, None)


def build_output_row(input_row: pd.Series, result: dict) -> dict:
    label = result['label']
    label_binary = 'benign' if label == 'benign' else 'adversarial'
    row = {'sample_id': input_row['sample_id']}
    for column in GROUND_TRUTH_COLUMNS:
        if column in input_row.index:
            row[column] = input_row[column]
    row.update({
        'llm_pred_binary': label_binary,
        'llm_pred_raw': label,
        'llm_pred_category': result['_category'],
        'llm_conf_binary': result['confidence'],
        'llm_evidence': result.get('evidence', ''),
        'llm_stages_run': 1,
        'llm_provider_name': result['_provider_name'],
        'llm_model_name': result['_model_name'],
        'llm_raw_response_text': result['_raw_response_text'],
        'llm_parse_success': result['_parse_success'],
        'clf_label': label,
        'clf_category': result['_category'],
        'clf_confidence': result['confidence'],
        'clf_evidence': result.get('evidence', ''),
        'clf_nlp_attack_type': result['nlp_attack_type'],
        'clf_provider_name': result['_provider_name'],
        'clf_model_name': result['_model_name'],
        'clf_raw_response_text': result['_raw_response_text'],
        'clf_parse_success': result['_parse_success'],
        'clf_token_logprobs': json.dumps(result.get('_token_logprobs')),
    })
    return row


def write_checkpoint(target: dict, rows: list[dict]) -> None:
    if not rows:
        return
    checkpoint = target['checkpoint_path']
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    new_df = pd.DataFrame(rows)
    if checkpoint.exists():
        existing_df = pd.read_parquet(checkpoint)
        out_df = pd.concat([existing_df, new_df], ignore_index=True)
        out_df = out_df.drop_duplicates(subset=['sample_id'], keep='last')
    else:
        out_df = new_df
    out_df.to_parquet(checkpoint, index=False)
    print(f"Checkpoint rows for {target['progress_label']}: {len(out_df):,} -> {checkpoint}")


def assert_valid_output(out_df: pd.DataFrame, target: dict) -> None:
    expected_columns = {'sample_id', *PREDICTION_COLUMNS}
    missing_columns = expected_columns - set(out_df.columns)
    if missing_columns:
        raise AssertionError(f"{target['progress_label']} missing expected columns: {sorted(missing_columns)}")
    assert not any(col.startswith('judge_') for col in out_df.columns), 'Output must not contain judge_* columns'
    assert valid_prediction_mask(out_df).all(), f"{target['progress_label']} contains invalid classifier prediction rows"
    assert set(out_df['llm_stages_run'].dropna().unique()) == {1}, 'Classifier-only outputs must have llm_stages_run=1'


In [ ]:
from tqdm.auto import tqdm


def load_main_target_df(target: dict) -> pd.DataFrame:
    split_path = target['input_path']
    if not split_path.exists():
        raise FileNotFoundError(f"Missing split for {target['progress_label']}: {split_path}")
    df = pd.read_parquet(split_path)
    missing = {target['text_col']} - set(df.columns)
    if missing:
        raise ValueError(f"{split_path} missing required columns: {sorted(missing)}")
    return df


def load_external_target_df(target: dict) -> pd.DataFrame:
    dataset_key = target['dataset_key']
    ds_cfg = cfg['external_datasets'][dataset_key]
    df = load_external_dataset(ds_cfg)
    missing = {target['text_col'], target['label_col']} - set(df.columns)
    if missing:
        raise ValueError(f"External dataset {dataset_key} missing required columns: {sorted(missing)}")
    return df


def prepare_target_df(target: dict) -> pd.DataFrame:
    if target['kind'] == 'main':
        df = load_main_target_df(target)
    elif target['kind'] == 'external':
        df = load_external_target_df(target)
    else:
        raise ValueError(f"Unknown target kind: {target['kind']}")

    df = df.copy()
    target_text_col = target['text_col']
    if TARGET_LIMIT is not None:
        df = df.head(int(TARGET_LIMIT)).copy()
    df['sample_id'] = df[target_text_col].apply(build_sample_id)
    return df


def run_target(target: dict) -> dict:
    print(f"\n=== {target['progress_label']} ===")
    eval_df = prepare_target_df(target)
    checkpoint_path = target['checkpoint_path']
    completed_ids = set()
    if checkpoint_path.exists():
        checkpoint_df = pd.read_parquet(checkpoint_path)
        valid_checkpoint_df = checkpoint_df[valid_prediction_mask(checkpoint_df)].copy()
        invalid_checkpoint_rows = len(checkpoint_df) - len(valid_checkpoint_df)
        if invalid_checkpoint_rows:
            print(f"Ignoring invalid checkpoint rows for {target['progress_label']}: {invalid_checkpoint_rows:,}")
        completed_ids = set(valid_checkpoint_df['sample_id'].tolist()) if 'sample_id' in valid_checkpoint_df.columns else set()
        print(f"Resuming {target['progress_label']} from checkpoint: {len(completed_ids):,} completed rows")

    pending_df = eval_df[~eval_df['sample_id'].isin(completed_ids)].reset_index(drop=True)
    print(f"Target {target['progress_label']}: total rows={len(eval_df):,}; completed rows={len(completed_ids):,}; pending rows={len(pending_df):,}")

    buffer: list[dict] = []
    for idx, input_row in tqdm(list(pending_df.iterrows()), total=len(pending_df), desc=f"Classifying {target['progress_label']}"):
        text = str(input_row[target['text_col']])
        try:
            result = classify_text(text)
        except Exception as exc:
            print(f"Row failed; writing parse-failure row for target={target['progress_label']} sample_id={input_row.sample_id}: {exc}")
            result = make_failure_result(raw_response_text=None)
        buffer.append(build_output_row(input_row, result))
        if len(buffer) >= CHECKPOINT_EVERY:
            write_checkpoint(target, buffer)
            buffer.clear()

    write_checkpoint(target, buffer)

    if checkpoint_path.exists():
        final_df = pd.read_parquet(checkpoint_path)
    else:
        final_df = pd.DataFrame(columns=['sample_id'] + PREDICTION_COLUMNS)

    pre_filter_rows = len(final_df)
    final_df = final_df[valid_prediction_mask(final_df)].copy()
    invalid_final_rows = pre_filter_rows - len(final_df)
    if invalid_final_rows:
        print(f"Dropped invalid final rows for {target['progress_label']}: {invalid_final_rows:,}")
    final_df = final_df.drop_duplicates(subset=['sample_id'], keep='last')
    target['output_path'].parent.mkdir(parents=True, exist_ok=True)
    final_df.to_parquet(target['output_path'], index=False)
    assert_valid_output(final_df, target)
    print(f"Final rows for {target['progress_label']}: {len(final_df):,} -> {target['output_path']}")
    return {
        'target': target['progress_label'],
        'rows': len(final_df),
        'output_path': str(target['output_path']),
        'failure_status': None,
    }


run_results = []
for target in TARGETS:
    try:
        run_results.append(run_target(target))
    except Exception as exc:
        failure_status = f'{type(exc).__name__}: {exc}'
        print(f"Target failed: {target['progress_label']} -> {failure_status}")
        run_results.append({
            'target': target['progress_label'],
            'rows': 0,
            'output_path': str(target['output_path']),
            'failure_status': failure_status,
        })

print('\nBatch output summary')
for result in run_results:
    status = result['failure_status'] or 'ok'
    print(f"- {result['target']}: {status}; rows={result['rows']:,}; output={result['output_path']}")

failed_results = [result for result in run_results if result['failure_status']]
if failed_results:
    raise RuntimeError(f"{len(failed_results)} target(s) failed; see Batch output summary above")


In [ ]:
for result in run_results:
    if result['failure_status']:
        continue
    target = next(item for item in TARGETS if item['progress_label'] == result['target'])
    out_df = pd.read_parquet(target['output_path'])
    assert_valid_output(out_df, target)
    print(f"Read back {len(out_df):,} rows from {target['output_path']}")
    print(out_df['llm_pred_binary'].value_counts(dropna=False))
    print(out_df[['llm_pred_raw', 'llm_conf_binary', 'llm_parse_success']].head())
